In [1]:
!pip install tensorflow numpy scipy pandas matplotlib scikit-learn openpyxl -q
print("✓ Done")

✓ Done


In [2]:
import os, shutil

for folder in ['ids_project/src', 'ids_project/data/raw',
               'ids_project/models', 'ids_project/results']:
    os.makedirs(folder, exist_ok=True)

os.chdir('ids_project')
print("✓ Working dir:", os.getcwd())

✓ Working dir: /content/ids_project


In [3]:
from google.colab import files
import shutil

print("Upload these 2 files:")
print("  - SWaT_Dataset_Normal_v1.xlsx")
print("  - SWaT_Dataset_Attack_v0.xlsx")

uploaded = files.upload()
for fname in uploaded:
    dest = f'data/raw/{fname}'
    shutil.move(fname, dest)
    print(f"✓ Saved: {dest}")

print("\nFiles in data/raw:", os.listdir('data/raw'))

Upload these 2 files:
  - SWaT_Dataset_Normal_v1.xlsx
  - SWaT_Dataset_Attack_v0.xlsx


Saving SWaT_Dataset_Normal_v1.xlsx to SWaT_Dataset_Normal_v1.xlsx
Saving SWaT_Dataset_Attack_v0.xlsx to SWaT_Dataset_Attack_v0.xlsx
✓ Saved: data/raw/SWaT_Dataset_Normal_v1.xlsx
✓ Saved: data/raw/SWaT_Dataset_Attack_v0.xlsx

Files in data/raw: ['SWaT_Dataset_Attack_v0.xlsx', 'SWaT_Dataset_Normal_v1.xlsx']


In [4]:
%%writefile src/__init__.py
# src package

Writing src/__init__.py


In [5]:
%%writefile src/preprocessing.py
import numpy as np
from scipy.signal import remez, lfilter
import pandas as pd

def design_fir_filter(passband_end=0.1, stopband_start=0.25, num_taps=11):
    return remez(num_taps, [0, passband_end, stopband_start, 1.0], [1, 0], fs=2)

def apply_fir_filter(signal, coeffs):
    return lfilter(coeffs, 1.0, signal)

def normalize_signal(signal):
    max_val = np.max(np.abs(signal))
    return (signal / max_val, max_val) if max_val != 0 else (signal, 1.0)

def create_ordered_pairs(signal, v):
    X, y = [], []
    for i in range(v, len(signal)):
        X.append(signal[i-v:i])
        y.append(signal[i])
    return np.array(X), np.array(y)

def split_dataset(X, y, train_ratio=0.70, val_ratio=0.10):
    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]
    n_tr = int(len(X) * train_ratio)
    n_va = int(len(X) * val_ratio)
    return ((X[:n_tr], y[:n_tr]),
            (X[n_tr:n_tr+n_va], y[n_tr:n_tr+n_va]),
            (X[n_tr+n_va:], y[n_tr+n_va:]))

def preprocess_signal(signal, v, fir_passband=0.1,
                      fir_stopband=0.25, fir_taps=11):
    coeffs            = design_fir_filter(fir_passband, fir_stopband, fir_taps)
    filtered          = apply_fir_filter(signal, coeffs)
    normalized, scale = normalize_signal(filtered)
    X, y              = create_ordered_pairs(normalized, v)
    return split_dataset(X, y), normalized, scale, coeffs

def load_swat_data(normal_path, attack_path,
                   excluded=('AIT201', 'AIT203', 'PIT502')):
    def _load(path):
        df = pd.read_excel(path, header=1, engine='openpyxl')
        df.columns = df.columns.astype(str).str.strip()
        return df

    df_n = _load(normal_path)
    df_a = _load(attack_path)

    print(f"  Normal : {df_n.shape}")
    print(f"  Attack : {df_a.shape}")
    print(f"  Columns: {list(df_n.columns[:8])} ...")

    # Extract labels
    labels    = None
    label_col = 'Normal/Attack'
    if label_col in df_a.columns:
        labels = (df_a[label_col].astype(str).str.strip().str.lower()
                  != 'normal').astype(int).values
        print(f"  Labels : {labels.sum()} attack / {len(labels)} total")
        df_a = df_a.drop(columns=[label_col])
    if label_col in df_n.columns:
        df_n = df_n.drop(columns=[label_col])

    # Drop timestamp
    for df in (df_n, df_a):
        ts = [c for c in df.columns
              if any(x in c.lower() for x in ('time','stamp','date'))]
        df.drop(columns=ts, inplace=True, errors='ignore')

    # Drop excluded sensors
    for col in excluded:
        df_n.drop(columns=[col], inplace=True, errors='ignore')
        df_a.drop(columns=[col], inplace=True, errors='ignore')

    # Keep common columns
    common = [c for c in df_n.columns if c in df_a.columns]
    df_n = df_n[common].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_a = df_a[common].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Drop constant columns
    varying = df_n.columns[df_n.std() > 0]
    df_n = df_n[varying]
    df_a = df_a[varying]

    # Remove first 6 hours stabilization
    if len(df_n) > 21600:
        df_n = df_n.iloc[21600:].reset_index(drop=True)
        print(f"  After 6h trim: {df_n.shape}")

    print(f"  Final sensors ({len(df_n.columns)}): {list(df_n.columns)}")
    return df_n, df_a, labels

Writing src/preprocessing.py


In [6]:
%%writefile src/model_builder.py
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import numpy as np
import itertools

def build_cnn_model(v, c, filters_per_layer, fs, pool_size, d1):
    inp = layers.Input(shape=(v, 1))
    x   = inp
    for block in range(c):
        f1 = filters_per_layer[2*block]
        f2 = filters_per_layer[2*block+1]
        x  = layers.Conv1D(f1, fs, activation='relu', padding='causal')(x)
        x  = layers.Conv1D(f2, fs, activation='relu', padding='causal')(x)
        x  = layers.MaxPooling1D(pool_size)(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(d1, activation='relu')(x)
    out = layers.Dense(1)(x)
    m   = models.Model(inp, out)
    m.compile(optimizer=optimizers.Adam(0.001), loss='mse')
    return m

def count_parameters(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))

def enumerate_configs_sorted():
    filters_c2 = [
        [4,8,8,16],[4,8,8,32],[4,8,16,32],
        [8,16,16,32],[8,16,32,64],[16,32,32,64]
    ]
    filters_c3 = [
        [4,8,8,16,16,32],[4,8,8,16,32,64],
        [8,16,16,32,32,64],[8,16,16,32,64,128]
    ]
    configs = []
    for v,c,fs,d1 in itertools.product([16,32,64],[2,3],[2,3,4],[30,40,50]):
        for filters in (filters_c2 if c==2 else filters_c3):
            cfg = dict(v=v,c=c,filters_per_layer=filters,
                       fs=fs,pool_size=2,d1=d1)
            try:
                tf.keras.backend.clear_session()
                m   = build_cnn_model(**cfg)
                cnt = count_parameters(m)
                configs.append((cfg, cnt))
            except:
                pass
    configs.sort(key=lambda x: x[1])
    return configs

Writing src/model_builder.py


In [7]:
%%writefile src/model_selector.py
import numpy as np
import tensorflow as tf
from src.preprocessing import create_ordered_pairs, split_dataset
from src.model_builder  import count_parameters, build_cnn_model

def compute_stats(y_true, y_pred):
    diff = np.abs(y_true - y_pred.flatten())
    return np.mean(diff), np.std(diff)

def criterion_1(mu_tr, mu_te, sig_tr, sig_te, tol=0.5):
    if mu_te == 0 or sig_te == 0:
        return False
    mu_pct  = abs(mu_te - mu_tr)  / abs(mu_te)  * 100
    sig_pct = abs(sig_te - sig_tr) / abs(sig_te) * 100
    return (mu_pct < tol) and (sig_pct < tol)

def compute_threshold(mu_te, sig_te):
    return abs(mu_te) + 3 * sig_te

def thresholding_algorithm(y_true, y_pred, T, z=15):
    y_pred  = y_pred.flatten()
    r_out   = np.zeros(len(y_true), dtype=int)
    counter = 0
    for i in range(len(y_true)):
        if abs(y_true[i] - y_pred[i]) > T:
            counter += 1
            if counter > z:
                r_out[i] = 1
        else:
            counter = 0
    return r_out

def criterion_2(sig_norm, model, v, T, z=15):
    X, y   = create_ordered_pairs(sig_norm, v)
    y_pred = model.predict(X[..., np.newaxis], verbose=0)
    return int(np.sum(thresholding_algorithm(y, y_pred, T, z))) == 0

def train_and_evaluate(cfg, train_data, val_data, test_data,
                       sig_norm, epochs=5, n_repeats=3, z=15, tol=0.5):
    X_tr, y_tr = train_data
    X_va, y_va = val_data
    X_te, y_te = test_data
    v = cfg['v']

    if X_tr.shape[1] != v:
        X, y = create_ordered_pairs(sig_norm, v)
        (X_tr,y_tr),(X_va,y_va),(X_te,y_te) = split_dataset(X, y)

    for attempt in range(n_repeats):
        tf.keras.backend.clear_session()
        model = build_cnn_model(**cfg)
        model.fit(X_tr[...,np.newaxis], y_tr,
                  validation_data=(X_va[...,np.newaxis], y_va),
                  epochs=epochs, batch_size=64, verbose=0)

        mu_tr,sig_tr = compute_stats(y_tr,
                           model.predict(X_tr[...,np.newaxis], verbose=0))
        mu_te,sig_te = compute_stats(y_te,
                           model.predict(X_te[...,np.newaxis], verbose=0))

        mu_pct  = abs(mu_te-mu_tr)  / max(abs(mu_te),  1e-9) * 100
        sig_pct = abs(sig_te-sig_tr) / max(abs(sig_te), 1e-9) * 100
        print(f"    attempt {attempt+1}: "
              f"mu={mu_pct:.2f}% sig={sig_pct:.2f}% (tol={tol}%)")

        if not criterion_1(mu_tr, mu_te, sig_tr, sig_te, tol):
            continue

        T = compute_threshold(mu_te, sig_te)
        if criterion_2(sig_norm, model, v, T, z):
            print(f"    ACCEPTED params={count_parameters(model)} T={T:.6f}")
            return model, T
        else:
            print(f"    crit-2 FAIL (false positives on normal data)")

    return None

def select_best_model(configs, train_data, val_data, test_data,
                      sig_norm, epochs=5, n_repeats=3, z=15, tol=0.5):
    total = len(configs)
    for i, (cfg, params) in enumerate(configs):
        print(f"  [{i+1}/{total}] params={params} v={cfg['v']} "
              f"c={cfg['c']} fs={cfg['fs']} d1={cfg['d1']}")
        result = train_and_evaluate(cfg, train_data, val_data, test_data,
                                    sig_norm, epochs, n_repeats, z, tol)
        if result is not None:
            return result, cfg
    print("  No suitable model found.")
    return None, None

Writing src/model_selector.py


In [8]:
%%writefile src/online_detector.py
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
from scipy.signal import lfilter

def run_online_detection(signal_attack, model, v, T, fir_coeffs, scale, z=15):
    filtered = lfilter(fir_coeffs, 1.0, signal_attack)
    y_norm   = filtered / scale
    X = np.array([y_norm[i-v:i]
                  for i in range(v, len(y_norm))])[..., np.newaxis]
    y_pred  = model.predict(X, verbose=0).flatten()
    y_true  = y_norm[v:]
    r_out   = np.zeros(len(y_norm), dtype=int)
    counter = 0
    for i in range(len(y_true)):
        if abs(y_true[i] - y_pred[i]) > T:
            counter += 1
            if counter > z:
                r_out[v+i] = 1
        else:
            counter = 0
    return r_out, y_norm, y_pred

def compute_metrics_per_sample(r_out, labels):
    n   = min(len(r_out), len(labels))
    r, l = r_out[:n], labels[:n]
    tp  = int(np.sum((r==1) & (l==1)))
    tn  = int(np.sum((r==0) & (l==0)))
    fp  = int(np.sum((r==1) & (l==0)))
    fn  = int(np.sum((r==0) & (l==1)))
    pre = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    rec = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1  = 2*pre*rec/(pre+rec) if (pre+rec) > 0 else 0.0
    acc = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else 0.0
    fpr = fp/(fp+tn) if (fp+tn) > 0 else 0.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn,
                precision=pre, recall=rec,
                f1=f1, accuracy=acc, fpr=fpr)

def plot_detection(y_norm, y_pred, r_out, v,
                   sensor_name='Sensor', save_path=None):
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(y_norm, color='steelblue', lw=0.8, label='Input signal')
    ax.plot(range(v, v+len(y_pred)), y_pred,
            color='crimson', lw=0.8, label='CNN prediction')
    alarm = np.where(r_out == 1)[0]
    if len(alarm):
        ax.axvspan(alarm[0], alarm[-1],
                   alpha=0.2, color='red', label='Alarm region')
    ax.set_title(f'Attack Detection - {sensor_name}')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Normalized amplitude')
    ax.legend(loc='upper right')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150)
    plt.close()

def print_metrics_table(m):
    print("\n" + "="*45)
    for k, v in m.items():
        if isinstance(v, float):
            print(f"  {k:<15} {v:.4f}")
        else:
            print(f"  {k:<15} {v}")
    print("="*45)

Writing src/online_detector.py


In [9]:
%%writefile main.py
import os, sys, argparse
import numpy as np
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from src.preprocessing   import preprocess_signal, load_swat_data
from src.model_builder   import enumerate_configs_sorted
from src.model_selector  import select_best_model
from src.online_detector import (run_online_detection,
                                  compute_metrics_per_sample,
                                  plot_detection,
                                  print_metrics_table)

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)

def run_swat(normal_path, attack_path):
    print("\n[SWaT] Loading dataset ...")
    df_n, df_a, labels = load_swat_data(normal_path, attack_path)
    configs     = enumerate_configs_sorted()
    all_results = {}

    n_samples = len(df_n)
    tol       = 0.5   # paper setting
    epochs    = 5     # paper setting
    print(f"  Samples: {n_samples} | tol={tol}% | epochs={epochs}")

    for col in df_n.columns:
        print(f"\n{'='*55}\n  Sensor: {col}\n{'='*55}")
        sig_normal = df_n[col].values.astype(float)

        if np.std(sig_normal) < 1e-6:
            print("  Skipping — constant signal")
            continue

        splits, sig_norm, scale, fir = preprocess_signal(sig_normal, v=16)
        print(f"  Train:{len(splits[0][0])} "
              f"Val:{len(splits[1][0])} "
              f"Test:{len(splits[2][0])}")

        result, best_cfg = select_best_model(
            configs, *splits, sig_norm,
            epochs=epochs, n_repeats=3, z=15, tol=tol)

        if result is None:
            print(f"  Skipping {col} — no model found")
            continue

        model, T = result
        model.save(f'models/{col}_model.keras')

        sig_attack = df_a[col].values.astype(float)
        r_out, y_norm_atk, y_pred_atk = run_online_detection(
            sig_attack, model, best_cfg['v'], T, fir, scale)

        if labels is not None:
            lab_bin = (labels != 0).astype(int)
            n       = min(len(r_out), len(lab_bin))
            metrics = compute_metrics_per_sample(r_out[:n], lab_bin[:n])
            print_metrics_table(metrics)
            all_results[col] = metrics

        plot_detection(y_norm_atk, y_pred_atk, r_out,
                       v=best_cfg['v'], sensor_name=col,
                       save_path=f'results/{col}_detection.png')

    print(f"\n{'='*55}")
    if all_results:
        f1s  = [v['f1']       for v in all_results.values()]
        accs = [v['accuracy'] for v in all_results.values()]
        fprs = [v['fpr']      for v in all_results.values()]
        recs = [v['recall']   for v in all_results.values()]
        print(f"  OVERALL RESULTS ({len(all_results)} sensors)")
        print(f"  Mean F1       : {np.mean(f1s):.4f}  (paper: 0.902)")
        print(f"  Mean Accuracy : {np.mean(accs)*100:.3f}%  "
              f"(paper: 97.846%)")
        print(f"  Mean FPR      : {np.mean(fprs)*100:.3f}%  "
              f"(paper: 0.135%)")
        print(f"  Mean Recall   : {np.mean(recs):.4f}  (paper: 0.830)")
    else:
        print("  No results — all sensors skipped")
    print(f"{'='*55}")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--mode',       default='swat')
    parser.add_argument('--normal_csv', default=None)
    parser.add_argument('--attack_csv', default=None)
    args = parser.parse_args()
    if args.mode == 'swat':
        run_swat(args.normal_csv, args.attack_csv)

Writing main.py


In [10]:
import shutil, sys, importlib
shutil.rmtree('src/__pycache__', ignore_errors=True)
sys.path.insert(0, '/content/ids_project')

import src.preprocessing as pp
importlib.reload(pp)

df_n, df_a, labels = pp.load_swat_data(
    'data/raw/SWaT_Dataset_Normal_v1.xlsx',
    'data/raw/SWaT_Dataset_Attack_v0.xlsx')

print(f"\n✓ Normal  : {df_n.shape}")
print(f"✓ Attack  : {df_a.shape}")
print(f"✓ Labels  : {labels.sum()} attack samples out of {len(labels)}")
print(f"✓ Sensors : {list(df_n.columns)}")
print("\n✓ Ready to run pipeline!")

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


  Normal : (495000, 53)
  Attack : (449919, 53)
  Columns: ['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202'] ...
  Labels : 54621 attack / 449919 total
  After 6h trim: (473400, 37)
  Final sensors (37): ['FIT101', 'LIT101', 'MV101', 'P101', 'AIT202', 'FIT201', 'MV201', 'P203', 'P205', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P402', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'PIT501', 'PIT503', 'FIT601', 'P602']

✓ Normal  : (473400, 37)
✓ Attack  : (449919, 37)
✓ Labels  : 54621 attack samples out of 449919
✓ Sensors : ['FIT101', 'LIT101', 'MV101', 'P101', 'AIT202', 'FIT201', 'MV201', 'P203', 'P205', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P402', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501

In [ ]:
!python main.py --mode swat \
  --normal_csv "data/raw/SWaT_Dataset_Normal_v1.xlsx" \
  --attack_csv "data/raw/SWaT_Dataset_Attack_v0.xlsx"

2026-04-26 16:55:41.995439: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777222542.017614   24840 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777222542.024787   24840 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777222542.044419   24840 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777222542.044468   24840 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777222542.044474   24840 computation_placer.cc:177] computation placer alr